# Component checks on the COMSOL reference fields

Cheap, training-free checks that give a **prior** for each small model component before the training notebook and the 4a/4b ablations: derivative backend, interior crop, Fourier modes and domain padding, training stride, and the Huber regime of the transient loss. The physics checks of the reference fields (residual floors, term sensitivity, water and sorption balances) stay here as supporting views.

- Every number comes from **published reference data of one membership** (`MEMBERSHIP`, Train by default, or validation). ID-test and OOD cases are refused, and every table names its membership and case count; each task shows its Dataset id once above its panel.
- The notebook trains nothing, writes nothing and changes no configuration. A section yields a prior for an ablation or a candidate range, never a decision: the criteria in the **Summary** panel are proposals while the component-check decision criteria are open.
- Each task has one lazy panel. Open it, choose a tab (cheap views first) and a view; a result is computed once and reused by later views and by the summary.

Analytic operator checks stay in `tests/domain/test_brinkman_residuals.py` and `tests/domain/test_physics_derivatives.py`. Run the notebook on a compute-node kernel ([Setup and execution](../docs/01_setup_and_execution.md#notebooks-on-compute-nodes)); the cost view times the CPU and, when the allocation has one, the GPU of that node.

In [ ]:
from functools import cache

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown
from IPython.display import display as show

from src import common
from src.analysis import component_checks as checks
from src.analysis import ui

STORAGE_ROOT = common.paths.get_storage_root()
STEADY_DATASET = "lentil-train1-val1-test1-pood1+chickpea-train1-val1-test1-pood1+kidney-bean-ood2__ref-param-ext__sobol-seed-9300"
TRANSIENT_DATASET = "lentil-train1-val1-test1-pood1+chickpea-train1-val1-test1-pood1+kidney-bean-ood2__ref-param-ext__sobol-seed-9200"
MEMBERSHIP = "train"  # train or validation; ID-test and OOD cases are refused
MAX_CASES: int | None = None  # leading membership cases per task; None uses all
MAP_CASE = 0  # case position shown in the residual maps
DECOMPOSITION_CROP = 2  # interior crop of the decomposition figure
TIMING_CROP = 2  # interior crop of the timed physics loss, as in the maintained physics configs

steady = checks.steady
spectral = checks.spectral
increments = checks.increments
transient = checks.transient
plots = checks.plots
scientific = "{:.2e}".format


def styled(frame, caption, formatter=scientific):
    """Return a table whose numeric columns are formatted, captioned with its membership and case count."""
    return frame.style.format(formatter, subset=list(frame.select_dtypes("number").columns), na_rep="n/a").set_caption(caption)


def section(entries, export_state):
    """Return one lazy dropdown section whose first view renders when its tab opens."""
    return ui.notebook.make_dropdown_section(entries, export_state=export_state, select_first=True)


def measured(result):
    """Return a cached result whose view was opened, else None, so the summary never starts a computation."""
    return result() if result.cache_info().currsize else None

## Steady flow

All terms use the domain scales from the inputs only: mean inlet pressure $\Delta p$, $G = \Delta p / L_y$ and Darcy velocity $U = K_g G / \mu$ with $K_g$ the geometric-mean permeability. The Brinkman length $\sqrt{K_g/\bar\varepsilon}$ against the grid spacing shows whether wall boundary layers are resolved. `fd` is the NeuralOp `FiniteDiff` (second-order central differences inside, third-order one-sided at the edges) and `fc` the spectral derivative with Fourier continuation (50 continuation points per axis). Precision `float32` is what training sees; `float64` uses exact float64 grids.

**1 Derivatives.** *Manufactured fields*: three smooth non-periodic analytic fields (polynomial times exponential, non-integer-period sines, a mixed product) on the reference grid and extent, with exact derivatives from autograd; the error of each backend per edge band and after crops 0, 1 and 2, in float32 and float64. Only these fields tell which backend is accurate; the reference residual and the FD-vs-FC difference cannot. *Reference floor*: the terms of the COMSOL solution, i.e. the floor below which a loss cannot distinguish a prediction from the reference:

| Column | Definition |
| --- | --- |
| `momentum` | training term: interior mean of $(R_x^2 + R_y^2)/G^2$, $R = -\nabla p + \nabla\cdot\tau - \mu K^{-1}u$, $\tau = (\mu/\varepsilon)(\nabla u + \nabla u^T - \tfrac{2}{3}(\nabla\cdot u) I)$ as in COMSOL |
| `trace_rms` | interior RMS of the trace part of $\nabla\cdot\tau$, $-\tfrac{2}{3}\nabla((\mu/\varepsilon)\nabla\cdot u)/G$ |
| `pointwise_divergence`, `porous_divergence` | interior mean of $(\nabla\cdot u\, L_y/U)^2$ and $(\nabla\cdot(\varepsilon u)\, L_y/(\bar\varepsilon U))^2$, removed from the loss |
| `flux_balance` | training term: mean over rows of $((q(y) - \bar q)/(U L_x))^2$ |
| `boundary` | training term: inlet mismatch plus squared outlet gauge, divided by $\Delta p^2$ |

The momentum decomposition shows pressure gradient and drag balancing each other; the viscous Brinkman term is small and its trace part smaller still (COMSOL's *Incompressible flow* stress keeps it; the loss uses the same stress). Band views use the cell distance to the nearest edge; the side walls are no-slip, so the stored velocity drops to zero over a Brinkman layer far thinner than one cell. *Cost*: median time and peak memory of forward plus backward of the dimensionless loss terms per backend at the training batch size (the selected cases tiled), float32, after 3 warm-ups over 5 synchronized repeats, with device, thread count and TF32 state. Without a GPU the CUDA rows read `unavailable`. The measurement is node-specific and is not persisted.

**2 Crop.** The case-median momentum floor per crop next to the case-median momentum ratio under a 1 % smooth error of $p, u, v$ (from the sensitivity below), for both backends at crops 0, 1 and 2.

**3 Sensitivity.** Each perturbation is added to the reference outputs and every loss term is divided by its unperturbed value (ratio 1: invisible). `smooth` adds a unit-RMS low-mode cosine field ($\le 4$ half-waves per axis), `noise` iid Gaussian values, both scaled to the stated fraction of each perturbed channel's RMS; `velocity_scale` multiplies $u, v$; `pressure_offset` adds a fraction of $\Delta p$. Seeds are explicit (`seed` column); float32.

**4 Modes.** Each output field, normalised as in training (the per-channel unit-Gaussian output normaliser fitted on the selected membership) without per-sample centring, is padded with the models' symmetric zero padding (0 or 0.1 per side), truncated to exactly the Fourier coefficients a neuraloperator spectral layer keeps, and cropped back: $n_y$ centred full-FFT modes along $y$ and $n_x/2 + 1$ rFFT modes along $x$. A nonzero case mean therefore meets the zero padding as a jump, as lifted features do. Curves vary one axis with the other complete; dotted lines mark the maintained `n_modes`. The relative L2 and edge-band errors (cells within 2 of an edge) are divided by the root mean square of the normalised field. This is a floor or prior only: the networks truncate lifted features, not the raw fields.


In [ ]:
reference = checks.sources.load_steady_reference(STEADY_DATASET, membership=MEMBERSHIP, storage_root=STORAGE_ROOT, max_cases=MAX_CASES)
steady_caption = checks.sources.membership_caption(reference.membership, reference.case_labels)
steady_config = checks.sources.maintained_base_config("steady_flow")
steady_modes = spectral.maintained_mode_settings("steady_flow")
show(Markdown(f"Dataset `{reference.dataset_id}`\n\n**{steady_caption}**"))


@cache
def residuals():
    """Return the reference residual floors per case, backend, precision and crop."""
    return steady.summarize_reference_residuals(reference)


@cache
def backend_fields():
    """Return the full-grid float32 residual fields per backend."""
    return steady.evaluate_backends(reference)


@cache
def sensitivity():
    """Return the perturbed-to-reference loss ratios of both backends at crops 0, 1 and 2."""
    return steady.evaluate_sensitivity(reference, settings=steady.CROP_SETTINGS)


@cache
def crop_table():
    """Return the momentum floor and 1 % smooth sensitivity per backend and crop."""
    return steady.crop_overview(residuals(), sensitivity())


@cache
def manufactured():
    """Return the backend errors against exact derivatives on the reference grid and extent."""
    return steady.manufactured_derivative_errors(reference.extent(0), tuple(reference.outputs.shape[-2:]))


@cache
def physics_cost():
    """Return the forward plus backward time of the physics loss at the training batch size."""
    batch = steady_config["data"]["batch_size"]
    copies = -(-batch // len(reference.case_ids))
    return checks.timing.time_physics_loss(
        reference.inputs.repeat(copies, 1, 1, 1)[:batch],
        reference.outputs.repeat(copies, 1, 1, 1)[:batch],
        input_fields=reference.input_fields,
        output_fields=reference.output_fields,
        backends=steady.DERIVATIVE_BACKENDS,
        crop=TIMING_CROP,
    )


@cache
def steady_truncation():
    """Return the truncation errors of every output field per case, mode count and padding."""
    return spectral.steady_truncation_errors(reference, steady_modes)

In [ ]:
def view_floor():
    """Show the median and maximum reference floor over cases."""
    show(styled(steady.residual_overview(residuals()), f"median over cases | {steady_caption}"))
    return styled(steady.residual_overview(residuals(), statistic="max"), f"maximum over cases | {steady_caption}")


def view_decomposition():
    """Show the momentum terms per backend and crop and their figure at the decomposition crop."""
    table = steady.momentum_term_table(backend_fields(), case_labels=reference.case_labels)
    show(styled(table.groupby(["backend", "crop"]).median(numeric_only=True), f"median over cases | {steady_caption}"))
    return plots.plot_term_decomposition(table[table["crop"] == DECOMPOSITION_CROP])


def view_bands():
    """Show the residual per boundary band and the profiles along each edge."""
    bands = steady.residual_band_table(backend_fields(), case_labels=reference.case_labels)
    show(styled(steady.band_overview(bands), f"median over cases | {steady_caption}"))
    return plots.plot_edge_profiles({label: steady.edge_profiles(steady.residual_magnitude(item)) for label, item in backend_fields().items()})


def view_maps():
    """Map the momentum residual of one case for both backends."""
    fields = backend_fields()
    label = reference.case_labels[MAP_CASE]
    return plots.plot_residual_maps([fields["fd"], fields["fc"]], case_index=MAP_CASE, case_label=label, extent=reference.extent(MAP_CASE))


def view_fd_fc():
    """Show the band difference of FC to FD derivatives on the reference fields."""
    comparison = steady.compare_derivatives(reference)
    show(styled(steady.derivative_overview(comparison), f"median over cases, float64 | {steady_caption}"))
    return plots.plot_derivative_difference(comparison[comparison["backend"] == "fc"])


def view_sensitivity():
    """Show the median and minimum ratio of every loss term and the ratio figure."""
    for term in ("momentum", "flux_balance", "boundary"):
        caption = f"{term}: perturbed / reference, median and minimum over cases and seeds | {steady_caption}"
        show(styled(steady.sensitivity_overview(sensitivity(), term=term), caption, "{:.3g}".format))
    return plots.plot_sensitivity(sensitivity())


def view_manufactured():
    """Return the worst manufactured-field error per backend, precision, band and crop."""
    return styled(steady.manufactured_overview(manufactured()), "worst over fields and directions | manufactured fields, reference grid")


def view_mode_pairs(errors, keys):
    """Return the worst errors at the maintained mode pairs for both paddings."""
    configured = errors[errors["axis"] == "config"]
    return configured.pivot_table(index=["config", *keys], columns="padding", values=["relative_l2", "edge_relative"], aggfunc="max")


steady_export = {}
steady_tabs = {
    "Cases and settings": [
        ("Selected cases and flow regime", lambda: styled(steady.flow_regime(reference), steady_caption), "flow_regime"),
        ("Maintained physics settings", steady.maintained_physics_settings, "physics_settings"),
        ("Maintained modes and padding", lambda: steady_modes, "mode_settings"),
    ],
    "1 Derivatives": [
        ("Manufactured fields: error against exact derivatives", view_manufactured, "manufactured"),
        ("Reference floor per backend, crop and precision", view_floor, "reference_floor"),
        ("Reference floor against crop", lambda: plots.plot_crop_sweep(residuals()), "crop_sweep"),
        ("Reference terms per case", lambda: plots.plot_reference_terms(residuals(), settings=(("fd", 0), ("fd", 2), ("fc", 2))), "terms"),
        ("Momentum decomposition", view_decomposition, "decomposition"),
        ("Boundary bands and edge profiles", view_bands, "edge_profiles"),
        ("Residual maps of one case", view_maps, "residual_maps"),
        ("FD against FC on the reference fields", view_fd_fc, "fd_fc"),
        ("Physics-loss cost per backend and device", lambda: styled(physics_cost(), f"node-specific, not persisted | {steady_caption}"), "cost"),
    ],
    "2 Crop": [
        ("Floor and 1 % smooth sensitivity per crop", lambda: styled(crop_table(), f"medians and extremes over cases | {steady_caption}"), "crop"),
    ],
    "3 Sensitivity": [("Loss-term ratios under controlled errors", view_sensitivity, "sensitivity")],
    "4 Modes": [
        (
            "Truncation curves per axis",
            lambda: plots.plot_mode_curves(steady_truncation(), steady_modes, title=f"Steady outputs | {steady_caption}"),
            "steady_modes",
        ),
        ("Maintained mode pairs", lambda: styled(view_mode_pairs(steady_truncation(), ["field"]), steady_caption), "steady_mode_pairs"),
    ],
}
show(
    ui.notebook.make_lazy_panel_with_tabs(
        [section(entries, steady_export) for entries in steady_tabs.values()],
        tab_titles=list(steady_tabs),
        open_btn_text="Steady flow - Open",
        export_state=steady_export,
    )
)


## Transient drying

The transient checks first fit the maintained transient scaling (`scaling.mode` of `base.yaml`) on the selected membership through the training service, then read its cases once from the canonical source HDF5 files. Stride, scaling mode and Huber $\beta$ come from the maintained `base.yaml`.

**5 Modes.** As in section 4, for the states and the one-step increments at the training stride (at most 16 evenly spaced stored times per case), normalised as in training with the fitted scaling (state encoding and increment scale) without per-sample centring. A floor or prior only: the networks truncate lifted features.

**6 Stride.** The views of strides 1, 2 and 4 are built with the transient runtime's endpoint-preserving index rule; a stride that would drop a grid endpoint is refused. Each view is interpolated bilinearly back to the full grid; the tables give the per-frame RMSE divided by the case's channel standard deviation (states) or RMS (increments), as the maximum over stored times, per case. This is a floor only for evaluation on the full grid.

**7 Increments.** Per channel, the scaled increments of the fitted scaling: RMS, magnitude quantiles (50, 90, 99 %, from a logarithmic histogram) and maximum, the fraction of exact zeros, the fraction above the maintained Huber $\beta$, and the channel's share of an equal-weight loss (mean Huber value of a zero-increment prediction). If no scaled increment exceeds $\beta$, Huber acts as MSE as long as residuals stay below the target magnitude: a prior for the Huber ablation. The channel shares are a prior for the transient loss channel weights only. The stored time steps are listed per case with a uniformity flag.

**8 Balances.** The Generation contract states the total water balance $\tfrac{d}{dt}(m_{w,gr} + m_{v,gas}) = \dot m_{v,in} - \dot m_{v,out}$. Its COMSOL global series live only in the source `case.h5`; the Dataset and the evaluation artifacts carry the fields, from which grain water and evaporation follow exactly up to quadrature, pore-gas vapour only with the maintained Magnus saturation pressure (COMSOL evaluates its own), and boundary fluxes only as their advective part. The stored states are hourly samples of an adaptive BDF integration; the transient template (SHA-256 `46e7112c…`) sets order 1–2 and relative tolerance 0.01, and neither setting is recorded in the case provenance. Hourly states cannot separate sub-hourly variation from solver error.

Closures are storage change minus integrated net source, summed over the case and divided by the water removed from the grain. Rules: trapezoidal (`_trapezoid`), left endpoint (`_left`) and, for the grain balance, Simpson. Rule differences bound only the quadrature error of smoothly varying rates; agreement between rules does not exclude sub-hourly variation, so a closure error is either sub-hourly dynamics or solver error. Field-versus-COMSOL columns give the largest relative difference over stored times. Inlet deviations compare the stored inlet row with the scheduled inlet state, separately for the interior of the inlet and its two wall corners. `negative_water_points` counts negative stored water (integrated as zero); `validity_invalid_points` counts reference values that the maintained evaluation validity rule (`T` range, $0 \le \varphi \le 1.10$, non-negative water, corner blocks excluded) would flag in a prediction. Sorption medians are RMS residuals of the two compartment equations divided by the RMS storage rate.

In [ ]:
transient_config = checks.sources.maintained_base_config("transient_drying")
transient_modes = spectral.maintained_mode_settings("transient_drying")
SPATIAL_STRIDE = transient_config["data"]["spatial_stride"]
HUBER_BETA = transient_config["loss"]["data"]["beta"]
transient_settings = pd.Series(
    {"spatial_stride": SPATIAL_STRIDE, "scaling.mode": transient_config["scaling"]["mode"], "huber_beta": HUBER_BETA}, name="base.yaml"
).to_frame()
show(Markdown(f"Dataset `{TRANSIENT_DATASET}`"))


@cache
def transient_scan():
    """Fit the scaling on the membership, then scan its cases once for modes, strides, increments and time steps."""
    scaling = increments.fit_membership_scaling(
        TRANSIENT_DATASET, membership=MEMBERSHIP, config=transient_config, storage_root=STORAGE_ROOT, max_cases=MAX_CASES
    )
    cases = checks.sources.iter_transient_reference_cases(TRANSIENT_DATASET, membership=MEMBERSHIP, storage_root=STORAGE_ROOT, max_cases=MAX_CASES)
    return increments.scan_transient_cases(cases, scaling=scaling, spatial_stride=SPATIAL_STRIDE, beta=HUBER_BETA, settings=transient_modes)


@cache
def balances():
    """Return the balance diagnostics of every membership case."""
    return transient.evaluate_dataset(TRANSIENT_DATASET, membership=MEMBERSHIP, storage_root=STORAGE_ROOT, max_cases=MAX_CASES)


def transient_caption():
    """Return the caption naming the scanned transient cases."""
    labels = [label for label in transient_scan().time_steps.index if label != "all cases"]
    return checks.sources.membership_caption(MEMBERSHIP, labels)


def view_transient_modes():
    """Plot the truncation curves of the states and increments at the training stride."""
    return plots.plot_mode_curves(transient_scan().spectral, transient_modes, title=f"States and increments | {transient_caption()}")


def view_stride():
    """Show refused strides, the worst floor per stride, and its per-case values."""
    floor = transient_scan().stride
    show(floor.loc[floor["status"] != "admissible", ["stride", "status"]].drop_duplicates("stride"))
    worst = floor.pivot_table(index=["stride", "quantity"], columns="field", values="max_normalized_rmse", aggfunc="max")
    show(styled(worst, f"maximum over cases and stored times | {transient_caption()}"))
    per_case = floor.pivot_table(index=["case", "stride"], columns=["quantity", "field"], values="max_normalized_rmse", aggfunc="max")
    return styled(per_case, f"maximum over stored times | {transient_caption()}")


def view_balances():
    """Show the headline closures of every case and the closure figure."""
    cases = balances()
    caption = checks.sources.membership_caption(MEMBERSHIP, [item.label for item in cases])
    show(transient.summary_table(cases).style.set_caption(caption))
    return plots.plot_transient_balances({item.label: item.balance for item in cases})


transient_export = {}
transient_tabs = {
    "Settings": [
        ("Maintained stride, scaling and Huber beta", lambda: transient_settings, "transient_settings"),
        ("Maintained modes and padding", lambda: transient_modes, "transient_mode_settings"),
        ("Where balance terms are persisted", transient.balance_term_availability, "balance_availability"),
    ],
    "5 Modes": [
        ("Truncation curves of states and increments", view_transient_modes, "transient_modes"),
        (
            "Maintained mode pairs",
            lambda: styled(view_mode_pairs(transient_scan().spectral, ["quantity", "field"]), transient_caption()),
            "transient_mode_pairs",
        ),
    ],
    "6 Stride": [("Full-grid floor per stride", view_stride, "stride")],
    "7 Increments": [
        ("Scaled increments and Huber regime", lambda: styled(transient_scan().increments, transient_caption(), "{:.3g}".format), "increments"),
        ("Stored time steps", lambda: styled(transient_scan().time_steps, transient_caption(), "{:.6g}".format), "time_steps"),
    ],
    "8 Balances": [
        ("Balance closures per case", view_balances, "balances"),
        ("Fields against COMSOL", lambda: plots.plot_field_vs_comsol({item.label: item.comparison for item in balances()}), "field_vs_comsol"),
        ("Sorption residuals", lambda: plots.plot_sorption_residuals({item.label: item.sorption for item in balances()}), "sorption"),
    ],
}
show(
    ui.notebook.make_lazy_panel_with_tabs(
        [section(entries, transient_export) for entries in transient_tabs.values()],
        tab_titles=list(transient_tabs),
        open_btn_text="Transient drying - Open",
        export_state=transient_export,
    )
)

## Summary

One row per component: the measured value, the proposed criterion and its status. The thresholds are named constants in `src/analysis/component_checks/component_checks_summary.py`, all **proposed while the component-check decision criteria are open**: FC stays a training arm if its crop-2 floor is at most FD's and its batch cost at most 2× FD's; the crop is the smallest whose 1 % smooth ratio reaches 100; the smallest mode count with error ≤ 1e-3 bounds the HPO range from below and ≤ 1e-4 counts as sufficient; padding has no effect if the edge-band errors differ by less than 10 %; a stride is admissible if its floor is ≤ 1e-3; Huber ≡ MSE is the prior if no scaled increment exceeds $\beta$. The summary uses only results already computed in the panels above; the others read *not measured*. Reopening the summary panel or tab recomputes the table from the results computed by then.

In [ ]:
class LiveSummary:
    """Criteria table that the panel recomputes whenever its view becomes visible again."""

    def __init__(self):
        """Render the table from the results computed so far."""
        self.output = widgets.Output()
        self.activate()

    def activate(self):
        """Apply the proposed criteria to every result computed so far and redraw the table."""
        table = checks.summary.summary_table(
            crop=measured(crop_table), timing=measured(physics_cost), steady_modes=measured(steady_truncation), transient=measured(transient_scan)
        )
        with self.output:
            self.output.clear_output(wait=True)
            show(table.style.set_caption(f"{MEMBERSHIP} membership | criteria {checks.summary.CRITERIA_STATE}"))

    def _ipython_display_(self):
        """Display the output area that holds the table."""
        show(self.output)


summary_export = {}
show(
    ui.notebook.make_lazy_panel_with_tabs(
        [section([("Criteria table", LiveSummary, "summary")], summary_export)],
        tab_titles=["Summary"],
        open_btn_text="Summary - Open",
        export_state=summary_export,
    )
)


## Not covered here

- Physics weights from term magnitudes, metric sensitivity, DataLoader throughput, and anything that needs training: these need Train/validation ablations or are environment benchmarks.
- Energy balance (latent heat, wall loss, inlet enthalpy): no heat-flux exports.
- Diffusive boundary fluxes and the COMSOL saturation concentration: not persisted.
- Sub-hourly dynamics: only hourly states are stored.
- Grid convergence and physical model validity (e.g. Forchheimer effects at the observed velocities): outside the scope of an implementation check.
